# Codelist Generation with OMOPy

This notebook covers the **`omopy.codelist`** module — the toolkit for building,
manipulating, and diagnosing concept codelists against an OMOP CDM.

Topics covered:
1. **Vocabulary search** — find standard concepts by keyword
2. **Hierarchy navigation** — descendants and ancestors
3. **Drug & ATC codes** — ingredient-based and ATC-based lookups
4. **Codelist operations** — union, intersect, compare
5. **Subsetting** — filter by domain, vocabulary, or presence in data
6. **Stratification** — break a codelist down by domain, vocabulary, or concept class
7. **Diagnostics** — code use summaries and orphan code detection
8. **Manual codelist creation** — build a `Codelist` from scratch

## Configuration

Set the path to your DuckDB file and the schema name here. All subsequent cells
reference these variables.

In [1]:
# -- Configuration --------------------------------------------------------
DB_PATH = "../data/synthea.duckdb"  # path to DuckDB file (relative to this notebook)
CDM_SCHEMA = "base"  # schema containing the OMOP CDM tables
# -------------------------------------------------------------------------

In [2]:
from omopy.codelist import (
    compare_codelists,
    get_ancestors,
    get_atc_codes,
    get_candidate_codes,
    get_descendants,
    get_drug_ingredient_codes,
    get_mappings,
    intersect_codelists,
    stratify_by_concept_class,
    stratify_by_domain,
    stratify_by_vocabulary,
    subset_by_domain,
    subset_by_vocabulary,
    subset_to_codes_in_use,
    summarise_code_use,
    summarise_orphan_codes,
    union_codelists,
)
from omopy.connector import cdm_from_con
from omopy.generics import Codelist

## Connect to CDM

We reuse the same Synthea DuckDB database from notebook 01. The CDM reference
object (`cdm`) is passed to every codelist function that needs vocabulary tables.

In [3]:
cdm = cdm_from_con(DB_PATH, cdm_schema=CDM_SCHEMA)
print(cdm)

CdmReference(name='dbt-synthea', version=5.4, source=duckdb, tables=36)


## Searching for Concepts

`get_candidate_codes` searches the vocabulary tables for concepts matching one or
more keywords. You can narrow results with optional filters:

```python
get_candidate_codes(
    cdm,
    ["sinusitis"],                # keywords (positional)
    standard_concept="S",         # "S" = Standard, "C" = Classification, None = any
    vocabulary_id=None,            # e.g. "SNOMED", "ICD10CM"
    concept_class_id=None,         # e.g. "Clinical Finding"
    domains=None,                  # e.g. "Condition", "Drug"
    exclude=None,                  # keywords to exclude
)
```

It returns a `Codelist` object.

### Basic keyword search

In [4]:
# Find all standard concepts matching "sinusitis"
sinusitis_codes = get_candidate_codes(
    cdm,
    ["sinusitis"],
    standard_concept="S",
)
print(sinusitis_codes)

Codelist(1 codelist(s), 3 total concept ID(s))


### Filtering by domain

In [5]:
# Restrict to the Condition domain
sinusitis_conditions = get_candidate_codes(
    cdm,
    ["sinusitis"],
    standard_concept="S",
    domains="Condition",
)
print(sinusitis_conditions)

Codelist(1 codelist(s), 3 total concept ID(s))


### Filtering by vocabulary and concept class

In [6]:
# SNOMED Clinical Findings only
sinusitis_snomed = get_candidate_codes(
    cdm,
    ["sinusitis"],
    standard_concept="S",
    vocabulary_id="SNOMED",
    concept_class_id="Clinical Finding",
)
print(sinusitis_snomed)

Codelist(1 codelist(s), 0 total concept ID(s))


### Excluding unwanted terms

In [7]:
# Search for sinusitis but exclude "chronic" concepts
sinusitis_no_chronic = get_candidate_codes(
    cdm,
    ["sinusitis"],
    standard_concept="S",
    exclude=["chronic"],
)
print(sinusitis_no_chronic)

Codelist(1 codelist(s), 2 total concept ID(s))


### Searching for hypertension

In [8]:
# Another common condition in Synthea — concept 320128 (Essential hypertension)
hypertension_codes = get_candidate_codes(
    cdm,
    ["hypertension"],
    standard_concept="S",
    domains="Condition",
)
print(hypertension_codes)

Codelist(1 codelist(s), 5 total concept ID(s))


### Getting source-to-standard mappings

`get_mappings` follows `concept_relationship` links to find source codes that map
to (or from) the concepts in a codelist.

In [9]:
# Find non-standard codes that map to our sinusitis codelist
sinusitis_mapped = get_mappings(
    cdm,
    sinusitis_conditions,
    relationship_id="Mapped from",
)
print(sinusitis_mapped)

Codelist(1 codelist(s), 3 total concept ID(s))


## Exploring Concept Hierarchies

OMOP vocabularies are organised in hierarchies. `get_descendants` walks *down* the
hierarchy; `get_ancestors` walks *up*.

Both accept a `Codelist`, a single concept ID (`int`), or a `list[int]`.

### Descendants

In [10]:
# Get all descendants of "Viral sinusitis" (concept_id 40481087)
viral_sinusitis_desc = get_descendants(cdm, concept_id=40481087)
print("Descendants of Viral sinusitis:")
print(viral_sinusitis_desc)

Descendants of Viral sinusitis:
Codelist(1 codelist(s), 1 total concept ID(s))


In [11]:
# Descendants from a codelist — expands every concept in the list
sinusitis_with_desc = get_descendants(cdm, concept_id=sinusitis_conditions)
print("Sinusitis conditions + descendants:")
print(sinusitis_with_desc)

Sinusitis conditions + descendants:
Codelist(1 codelist(s), 3 total concept ID(s))


### Ancestors

In [12]:
# Walk up the hierarchy from "Viral sinusitis"
viral_sinusitis_anc = get_ancestors(cdm, concept_id=40481087)
print("Ancestors of Viral sinusitis:")
print(viral_sinusitis_anc)

Ancestors of Viral sinusitis:
Codelist(1 codelist(s), 31 total concept ID(s))


In [13]:
# Ancestors from a list of concept IDs
ancestors = get_ancestors(cdm, concept_id=[40481087, 320128])
print("Ancestors of Viral sinusitis + Essential hypertension:")
print(ancestors)

Ancestors of Viral sinusitis + Essential hypertension:
Codelist(1 codelist(s), 35 total concept ID(s))


## Drug & ATC Codes

Two specialised helpers make it easy to build drug codelists:

- `get_drug_ingredient_codes(cdm, ingredient=...)` — finds all drug concepts
  linked to an ingredient name.
- `get_atc_codes(cdm, atc_name=..., level=...)` — finds concepts within an ATC
  classification level (levels `"1"` through `"5"`).

In [14]:
# All drug concepts containing acetaminophen as an ingredient
acetaminophen = get_drug_ingredient_codes(cdm, ingredient="acetaminophen")
print("Acetaminophen drug codes:")
print(acetaminophen)

Acetaminophen drug codes:
Codelist(1 codelist(s), 18 total concept ID(s))


In [15]:
# ATC level 1 — broad therapeutic category
alimentary_atc = get_atc_codes(cdm, atc_name="alimentary", level="1")
print("ATC level 1 — alimentary:")
print(alimentary_atc)

ATC level 1 — alimentary:
Codelist(0 codelist(s), 0 total concept ID(s))


In [16]:
# ATC level 3 for a more specific grouping
analgesics_atc = get_atc_codes(cdm, atc_name="analgesics", level="3")
print("ATC level 3 — analgesics:")
print(analgesics_atc)

ATC level 3 — analgesics:
Codelist(0 codelist(s), 0 total concept ID(s))


## Codelist Operations

Codelists can be combined and compared with set-like operations.

### Union

`union_codelists` merges two or more codelists into one, combining all concept IDs.

In [17]:
combined = union_codelists(sinusitis_conditions, hypertension_codes)
print("Union of sinusitis + hypertension:")
print(combined)

Union of sinusitis + hypertension:
Codelist(2 codelist(s), 8 total concept ID(s))


### Intersection

`intersect_codelists` keeps only concept IDs present in *all* provided codelists.

In [18]:
overlap = intersect_codelists(sinusitis_codes, sinusitis_conditions)
print("Intersection (sinusitis all vs. conditions only):")
print(overlap)

Intersection (sinusitis all vs. conditions only):
Codelist(1 codelist(s), 3 total concept ID(s))


### Compare

`compare_codelists` shows which concepts are shared and which are unique to each
codelist — useful for auditing. Returns a `dict[str, dict[str, list[int]]]`.

In [19]:
comparison = compare_codelists(sinusitis_codes, sinusitis_conditions)
print(comparison)

{'sinusitis': {'only_a': [], 'only_b': [], 'both': [257012, 4283893, 40481087]}}


## Subsetting Codelists

Subsetting functions *narrow* a codelist by some criterion. Note the argument
order: `(codelist, cdm, ...)`.

### Subset by domain

In [20]:
# Keep only Condition-domain concepts from the combined codelist
conditions_only = subset_by_domain(combined, cdm, "Condition")
print("Subset by domain = 'Condition':")
print(conditions_only)

Subset by domain = 'Condition':
Codelist(2 codelist(s), 8 total concept ID(s))


### Subset by vocabulary

In [21]:
# Keep only SNOMED concepts
snomed_only = subset_by_vocabulary(combined, cdm, "SNOMED")
print("Subset by vocabulary = 'SNOMED':")
print(snomed_only)

Subset by vocabulary = 'SNOMED':
Codelist(2 codelist(s), 8 total concept ID(s))


### Subset to codes in use

This is particularly useful — it filters a codelist to only those concept IDs
that actually appear in the clinical tables of the CDM.

In [22]:
# Which sinusitis codes actually appear in the Synthea data?
in_use = subset_to_codes_in_use(sinusitis_conditions, cdm)
print("Sinusitis codes present in Synthea data:")
print(in_use)

Sinusitis codes present in Synthea data:
Codelist(1 codelist(s), 2 total concept ID(s))


## Stratifying Codelists

Stratification functions split a codelist into groups without discarding any
concepts. They return a dictionary (or similar structure) keyed by the
stratification variable.

### Stratify by domain

In [23]:
by_domain = stratify_by_domain(combined, cdm)
print("Stratified by domain:")
print(by_domain)

Stratified by domain:
Codelist(2 codelist(s), 8 total concept ID(s))


### Stratify by vocabulary

In [24]:
by_vocab = stratify_by_vocabulary(combined, cdm)
print("Stratified by vocabulary:")
print(by_vocab)

Stratified by vocabulary:
Codelist(2 codelist(s), 8 total concept ID(s))


### Stratify by concept class

In [25]:
by_class = stratify_by_concept_class(combined, cdm)
print("Stratified by concept class:")
print(by_class)

Stratified by concept class:
Codelist(2 codelist(s), 8 total concept ID(s))


## Codelist Diagnostics

Diagnostic functions help you understand how well your codelist captures the data
in the CDM. Both return a polars `DataFrame`.

### Summarise code use

`summarise_code_use` counts how often each concept in the codelist appears across
the clinical tables. Returns a polars `DataFrame`.

In [26]:
code_use = summarise_code_use(sinusitis_conditions, cdm)
print(code_use)

shape: (3, 6)
┌──────────────────┬────────────┬───────────────────┬───────────┬───────────────┬───────┐
│ concept_set_name ┆ concept_id ┆ concept_name      ┆ domain_id ┆ vocabulary_id ┆ count │
│ ---              ┆ ---        ┆ ---               ┆ ---       ┆ ---           ┆ ---   │
│ str              ┆ i64        ┆ str               ┆ str       ┆ str           ┆ i64   │
╞══════════════════╪════════════╪═══════════════════╪═══════════╪═══════════════╪═══════╡
│ sinusitis        ┆ 257012     ┆ Chronic sinusitis ┆ Condition ┆ SNOMED        ┆ 5     │
│ sinusitis        ┆ 40481087   ┆ Viral sinusitis   ┆ Condition ┆ SNOMED        ┆ 4     │
│ sinusitis        ┆ 4283893    ┆ Sinusitis         ┆ Condition ┆ SNOMED        ┆ 0     │
└──────────────────┴────────────┴───────────────────┴───────────┴───────────────┴───────┘


### Summarise orphan codes

`summarise_orphan_codes` identifies concepts in the codelist that have *no*
records in any clinical table — potential gaps or unused codes. Returns a polars
`DataFrame`.

In [27]:
orphans = summarise_orphan_codes(sinusitis_conditions, cdm)
print(orphans)

shape: (0, 6)
┌──────────────────┬────────────┬──────────────┬───────────┬──────────────┬───────┐
│ concept_set_name ┆ concept_id ┆ concept_name ┆ domain_id ┆ relationship ┆ count │
│ ---              ┆ ---        ┆ ---          ┆ ---       ┆ ---          ┆ ---   │
│ str              ┆ i64        ┆ str          ┆ str       ┆ str          ┆ i64   │
╞══════════════════╪════════════╪══════════════╪═══════════╪══════════════╪═══════╡
└──────────────────┴────────────┴──────────────┴───────────┴──────────────┴───────┘


## Manual Codelist Creation

`Codelist` is a `dict` subclass. Each key is a codelist name and the value is a
list of concept IDs. You can build one manually when you know the exact concepts
you need.

Useful properties:
- `.names` — list of codelist names (keys)
- `.all_concept_ids` — flat collection of every concept ID across all names

In [28]:
# Build a codelist by hand
my_codelist = Codelist(
    {
        "viral_sinusitis": [40481087],
        "essential_hypertension": [320128],
    }
)

print("Names:", my_codelist.names)
print("All concept IDs:", my_codelist.all_concept_ids)
print()
print(my_codelist)

Names: ['viral_sinusitis', 'essential_hypertension']
All concept IDs: {320128, 40481087}

Codelist(2 codelist(s), 2 total concept ID(s))


In [29]:
# Manual codelists work with every function shown above
my_descendants = get_descendants(cdm, concept_id=my_codelist)
print("Descendants of manual codelist:")
print(my_descendants)

Descendants of manual codelist:
Codelist(1 codelist(s), 2 total concept ID(s))


In [30]:
# Check which of our hand-picked codes are actually in the data
my_in_use = subset_to_codes_in_use(my_codelist, cdm)
print("Manual codelist — codes in use:")
print(my_in_use)

Manual codelist — codes in use:
Codelist(2 codelist(s), 2 total concept ID(s))


## Summary

This notebook demonstrated the full `omopy.codelist` workflow:

| Step | Function(s) |
|---|---|
| **Search** | `get_candidate_codes`, `get_mappings` |
| **Navigate hierarchy** | `get_descendants`, `get_ancestors` |
| **Drug lookups** | `get_drug_ingredient_codes`, `get_atc_codes` |
| **Combine** | `union_codelists`, `intersect_codelists`, `compare_codelists` |
| **Subset** | `subset_by_domain`, `subset_by_vocabulary`, `subset_to_codes_in_use` |
| **Stratify** | `stratify_by_domain`, `stratify_by_vocabulary`, `stratify_by_concept_class` |
| **Diagnose** | `summarise_code_use`, `summarise_orphan_codes` |
| **Manual** | `Codelist({...})` |

### Next steps

- Use these codelists with `omopy.cohort` to define study cohorts.
- Combine with `omopy.generics` subsetting to focus on specific patient populations.
- Export codelists for review or sharing with clinical collaborators.